# 🎯 Data Mining & Machine Learning Project
## Customer Churn Prediction — Telecommunications Sector

---

**Course:** Introduction to Data Mining and Data Science  
**Professor:** Dr. Yosra Hajjaji  
**Dataset:** IBM Telco Customer Churn (Kaggle)  
**Methodology:** CRISP-DM  
**Year:** 2025–2026

---

## 📌 Project Objective

This project applies the concepts from **Chapters 1 and 2** of the course:
- **Chapter 1:** Introduction to Data Mining and Decision Making
- **Chapter 2:** Introduction to Machine Learning and the Data Science Project (CRISP-DM)

The use case is **customer churn prediction** in a telecommunications company:  
> *"Which customers are likely to cancel their subscription in the coming months?"*

This problem directly mirrors the CRISP-DM example seen in class (*unpaid phone bills*).

---
# 📖 PART 1 — Foundations (Chapter 1)
## Introduction to Data Mining and Decision Making

### 1.1 Why Data Mining?

Telecommunications companies generate **massive volumes of data** every day:
- Call history, SMS, internet usage
- Monthly billing, payment history
- Contract information (contract type, customer tenure)

Alone, this raw data has no value. **Data Mining** transforms it into **actionable knowledge**:

```
Raw Data  →  Information  →  Knowledge  →  Decision
```

In our project:  
- **Data:** 7,043 customers, 20 variables (contract, charges, services...)
- **Information:** Customers on monthly contracts pay on average €65/month
- **Knowledge:** Customers on monthly contract + fiber optic + no online security have 45% churn probability
- **Decision:** Launch a targeted retention campaign for this segment

### 1.2 Positioning in the Data Ecosystem

```
Big Data
  └── Business Intelligence (reporting, dashboards)
        └── Data Mining (pattern discovery)
              └── Machine Learning (predictive models)
                    └── Data Science (full scientific approach)
```

Our project covers **the entire chain**: from raw data all the way to a deployed production model.

### 1.3 Data Types in Our Dataset

| Type | Examples in our dataset | Treatment |
|------|------------------------|-----------|
| **Numerical** | tenure, MonthlyCharges, TotalCharges | StandardScaler |
| **Categorical** | Contract, InternetService, PaymentMethod | One-Hot Encoding |
| **Binary** | Partner, Dependents, PhoneService | 0/1 encoding |
| **Target (label)** | Churn (Yes/No) | 0/1 encoding |

### 1.4 Real-World Data Problems (Chapter 1 — "Real-World Data")

As covered in class, real-world data is never clean. Our dataset has all three problem types:

| Problem Type | Example in our dataset | Cause |
|-------------|----------------------|-------|
| **Incomplete** | 11 missing values in `TotalCharges` | Customer joined mid-cycle |
| **Noisy** | `TotalCharges` stored as string `"786.0"` | System export error |
| **Inconsistent** | `Churn` coded as `Yes/No` instead of `1/0` | Human input convention |

---
# 📖 PART 2 — CRISP-DM Methodology (Chapter 2)

We follow the **CRISP-DM** methodology (Cross Industry Standard Process for Data Mining), the most widely used framework in Data Science (49% of projects according to KDnuggets).

```
Phase 1 : Business Understanding  →  Define the business objective
Phase 2 : Data Understanding      →  Explore and understand the data
Phase 3 : Data Preparation        →  Cleaning, transformation, feature engineering
Phase 4 : Modeling                →  Train 4 ML models
Phase 5 : Evaluation              →  Compare and select best model
Phase 6 : Deployment              →  FastAPI + Streamlit dashboard
```

---
## ⚙️ Import Libraries

In [ ]:
# Uncomment if needed
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm shap mlflow joblib loguru

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── ML imports ────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve
)
import shap
import joblib
from pathlib import Path

# ── Plot settings ─────────────────────────────────────────────────────────────
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

print('✅ All libraries imported successfully')

---
# 🔵 PHASE 1 — Business Understanding

### Context
A telecommunications company loses customers every month (**churn**). Each lost customer represents a direct revenue loss. It is far less expensive to **retain an existing customer** than to acquire a new one.

### Business Objective
> Identify, **in advance**, customers likely to cancel their subscription in order to trigger targeted retention actions (commercial offer, customer call, pricing discount).

### Translation into a Data Mining Problem
| Dimension | Detail |
|-----------|--------|
| **Problem type** | Supervised binary classification |
| **Target variable** | Churn: Yes (1) / No (0) |
| **Success criteria** | ROC-AUC > 0.75, F1-score > 0.55 |
| **Model use** | Monthly churn risk score per customer |
| **Triggered action** | Retention campaign if probability > 35% |

### Business Hypothesis
As in the CRISP-DM example from class (unpaid phone bills), we assume that **customers about to churn change their behavior** before leaving:
- Short-term contract + minimal services = imminent departure signal
- High monthly charges without long-term commitment = dissatisfied customer
- New customer without add-ons = high attrition risk

---
# 🔵 PHASE 2 — Data Understanding

## Exploring the Dataset

In [ ]:
# ── Load the dataset ──────────────────────────────────────────────────────────
DATA_PATH = Path('../data/telco_churn.csv')   # adjust if needed
if not DATA_PATH.exists():
    DATA_PATH = Path('data/telco_churn.csv')

df = pd.read_csv(DATA_PATH)

print(f'✅ Dataset loaded: {df.shape[0]:,} customers × {df.shape[1]} variables')
df.head()

In [ ]:
# ── General overview ──────────────────────────────────────────────────────────
print('=' * 55)
print('DATASET SUMMARY')
print('=' * 55)
print(f'Number of customers  : {len(df):,}')
print(f'Number of variables  : {df.shape[1]}')
print(f'Real churn rate      : {(df["Churn"]=="Yes").mean():.1%}')
print(f'Missing values       :\n{df.isnull().sum()[df.isnull().sum()>0]}')
print('\nData types:')
print(df.dtypes.value_counts())

In [ ]:
# ── 2.1 Target variable distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

churn_counts = df['Churn'].value_counts()
axes[0].pie(
    churn_counts.values,
    labels=['No Churn', 'Churn'],
    colors=['#3498db', '#e74c3c'],
    autopct='%1.1f%%',
    startangle=90,
    explode=(0, 0.05)
)
axes[0].set_title('Churn / No Churn Distribution', fontsize=14, fontweight='bold')

contract_churn = df.groupby(['Contract', 'Churn']).size().unstack()
contract_churn.plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'], edgecolor='white')
axes[1].set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Contract Type')
axes[1].set_ylabel('Number of Customers')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(['No Churn', 'Churn'])

plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Month-to-month customers churn much more than yearly/two-year contract customers.')

In [ ]:
# ── 2.2 Numerical variables analysis ─────────────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
titles   = ['Tenure (months)', 'Monthly Charges (€)', 'Total Charges (€)']

for ax, col, title in zip(axes, num_cols, titles):
    churned     = df[df['Churn'] == 'Yes'][col].dropna()
    not_churned = df[df['Churn'] == 'No'][col].dropna()
    ax.hist(not_churned, bins=30, alpha=0.6, color='#3498db', label='No Churn', density=True)
    ax.hist(churned,     bins=30, alpha=0.6, color='#e74c3c', label='Churn',    density=True)
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.set_ylabel('Density')

plt.suptitle('Distribution of Numerical Variables by Churn Status', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Churned customers have shorter tenure and higher monthly charges.')

In [ ]:
# ── 2.3 Correlation matrix ────────────────────────────────────────────────────
df_num = df[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
df_num['TotalCharges'] = pd.to_numeric(df_num['TotalCharges'], errors='coerce')
df_num['Churn_num'] = (df['Churn'] == 'Yes').astype(int)

plt.figure(figsize=(8, 6))
sns.heatmap(
    df_num.corr(),
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    square=True, linewidths=0.5
)
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Churn is negatively correlated with tenure — the longer a customer stays, the less likely they are to churn.')

In [ ]:
# ── 2.4 Categorical variables analysis ───────────────────────────────────────
cat_cols = ['InternetService', 'PaymentMethod', 'SeniorCitizen']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, cat_cols):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    churn_rate.sort_values(ascending=True).plot(
        kind='barh', ax=ax, color='#e74c3c', edgecolor='white'
    )
    ax.set_title(f'Churn Rate by\n{col}', fontweight='bold')
    ax.set_xlabel('Churn Rate (%)')
    for i, v in enumerate(churn_rate.sort_values(ascending=True)):
        ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('categorical_churn.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Fiber optic internet and electronic check payment method are strongly associated with churn.')

### 📊 Phase 2 Summary — Data Understanding

Following the course framework (Chapter 1 — Real-World Data), we identified all three problem types:

| Problem Detected | Description | Planned Fix |
|-----------------|-------------|-------------|
| **Incomplete data** | 11 missing values in `TotalCharges` | Median imputation |
| **Noisy data** | `TotalCharges` stored as string type | Float type coercion |
| **Class imbalance** | 26% churn vs 74% non-churn | Optimal threshold on validation set |
| **Useless variable** | `customerID` has zero predictive value | Drop column |

---
# 🔵 PHASE 3 — Data Preparation

## Cleaning, Transformation & Feature Engineering

This phase directly corresponds to **Step 2: Scrub the Dirt** from the Data Cleaning guide:
- Missing Data → Imputation
- Data Type Issues → Type coercion
- Inconsistent Data → Uniform encoding
- Feature Engineering → New business-logic variables

In [ ]:
# ── 3.1 Data Cleaning ─────────────────────────────────────────────────────────
df_clean = df.copy()

# Drop customer ID (no predictive value)
df_clean.drop(columns=['customerID'], inplace=True)

# Fix data type: TotalCharges string → float
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

# Impute missing values with median (robust to outliers)
df_clean['TotalCharges'].fillna(df_clean['TotalCharges'].median(), inplace=True)

# Encode target variable: Yes/No → 1/0
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

print('✅ Cleaning complete')
print(f'Remaining missing values: {df_clean.isnull().sum().sum()}')
print(f'Dataset shape: {df_clean.shape}')

In [ ]:
# ── 3.2 Feature Engineering — CRM Business Variables ─────────────────────────
# We create 7 new variables derived from domain knowledge

# 1. Average cost per tenure month — high early spend signals dissatisfaction
df_clean['ChargesPerTenureMonth'] = df_clean['MonthlyCharges'] / (df_clean['tenure'] + 1)

# 2. New customer flag (< 6 months) — new customers churn more
df_clean['IsNewCustomer'] = (df_clean['tenure'] <= 6).astype(int)

# 3. Long-term customer flag (> 24 months) — loyal customers are stickier
df_clean['IsLongTerm'] = (df_clean['tenure'] >= 24).astype(int)

# 4. Service bundle score — more services = harder to leave
service_cols = ['PhoneService', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_clean['ServiceBundleScore'] = df_clean[service_cols].apply(
    lambda x: (x == 'Yes').sum(), axis=1
)

# 5. Contract risk score — encodes contract danger level
contract_risk = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
df_clean['ContractRiskScore'] = df_clean['Contract'].map(contract_risk)

# 6. Total charges gap — gap between expected and actual total charges
expected_total = df_clean['MonthlyCharges'] * df_clean['tenure']
df_clean['TotalChargesGap'] = expected_total - df_clean['TotalCharges']

# 7. High-risk combination flag: month-to-month + fiber + no security
df_clean['HighRiskCombo'] = (
    (df_clean['Contract'] == 'Month-to-month') &
    (df_clean['InternetService'] == 'Fiber optic') &
    (df_clean['OnlineSecurity'] == 'No')
).astype(int)

print('✅ Feature Engineering complete — 7 new variables created:')
new_features = ['ChargesPerTenureMonth', 'IsNewCustomer', 'IsLongTerm',
                'ServiceBundleScore', 'ContractRiskScore', 'TotalChargesGap', 'HighRiskCombo']
for f in new_features:
    print(f'   • {f}')

In [ ]:
# ── 3.3 Encoding + sklearn Pipeline setup ────────────────────────────────────
X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']

# Identify numerical and categorical columns
numerical_cols   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f'Numerical variables  ({len(numerical_cols)}): {numerical_cols}')
print(f'Categorical variables ({len(categorical_cols)}): {categorical_cols}')

# Preprocessing pipeline
# - Numerical: StandardScaler (mean=0, std=1) → so tenure doesn't dominate MonthlyCharges
# - Categorical: One-Hot Encoding → convert text categories to binary columns
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

In [ ]:
# ── 3.4 Stratified Train / Validation / Test split — 70 / 10 / 20 ────────────
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.125, random_state=42, stratify=y_train_val
)

print('✅ Stratified split complete (same churn rate preserved in each split):')
print(f'   Train      : {len(X_train):,} customers ({len(X_train)/len(X)*100:.0f}%) — churn rate: {y_train.mean():.1%}')
print(f'   Validation : {len(X_val):,} customers ({len(X_val)/len(X)*100:.0f}%) — churn rate: {y_val.mean():.1%}')
print(f'   Test       : {len(X_test):,} customers ({len(X_test)/len(X)*100:.0f}%) — churn rate: {y_test.mean():.1%}')

---
# 🔵 PHASE 4 — Modeling

## Machine Learning — Training 4 Models

As covered in class (Chapter 2 — Types of Machine Learning), this is a **supervised learning classification** problem:
- The data is **labeled** (Churn = Yes/No is already known for historical customers)
- The goal is to **predict the category** for new/future customers

We train 4 models for comparison:

| Model | Type | Characteristic |
|-------|------|----------------|
| **Logistic Regression** | Linear | Simple, interpretable, fast |
| **Random Forest** | Ensemble (bagging) | Robust, handles non-linearities |
| **XGBoost** | Ensemble (boosting) | High performance, widely used in industry |
| **LightGBM** | Ensemble (boosting) | Fast on large datasets |

In [ ]:
# ── 4.1 Define models ─────────────────────────────────────────────────────────
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            n_estimators=100, random_state=42,
            eval_metric='logloss', verbosity=0
        ))
    ]),
    'LightGBM': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LGBMClassifier(n_estimators=100, random_state=42, verbose=-1))
    ]),
}

print(f'✅ {len(models)} models defined and ready to train')

In [ ]:
# ── 4.2 Train and evaluate all models ────────────────────────────────────────
results = {}
trained_models = {}

for name, pipeline in models.items():
    print(f'🔄 Training: {name}...')
    pipeline.fit(X_train, y_train)
    
    # Validation set predictions
    y_val_proba = pipeline.predict_proba(X_val)[:, 1]
    y_val_pred  = (y_val_proba >= 0.5).astype(int)
    
    # Test set predictions
    y_test_proba = pipeline.predict_proba(X_test)[:, 1]
    y_test_pred  = (y_test_proba >= 0.5).astype(int)
    
    results[name] = {
        'Val ROC-AUC'    : round(roc_auc_score(y_val,  y_val_proba),  4),
        'Test ROC-AUC'   : round(roc_auc_score(y_test, y_test_proba), 4),
        'Test F1'        : round(f1_score(y_test,       y_test_pred),  4),
        'Test Precision' : round(precision_score(y_test, y_test_pred), 4),
        'Test Recall'    : round(recall_score(y_test,   y_test_pred),  4),
    }
    trained_models[name] = pipeline
    print(f'   ✅ Test ROC-AUC = {results[name]["Test ROC-AUC"]} | F1 = {results[name]["Test F1"]}')

results_df = pd.DataFrame(results).T
print('\n' + '='*60)
print('MODEL COMPARISON TABLE')
print('='*60)
print(results_df.to_string())

---
# 🔵 PHASE 5 — Evaluation

## Evaluating and Selecting the Best Model

In [ ]:
# ── 5.1 ROC Curves — all 4 models ────────────────────────────────────────────
plt.figure(figsize=(10, 7))

for (name, pipeline), color in zip(trained_models.items(), COLORS):
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random model (AUC = 0.5)')
plt.fill_between([0,1],[0,1],[1,1], alpha=0.05, color='green')
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('ROC Curves — Comparison of 4 Models\n(Green area = better performance)',
          fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.2 Select the best model ────────────────────────────────────────────────
best_model_name = results_df['Test ROC-AUC'].idxmax()
best_model      = trained_models[best_model_name]

print(f'🏆 Best model: {best_model_name}')
print(f'   ROC-AUC   = {results_df.loc[best_model_name, "Test ROC-AUC"]}')
print(f'   F1-Score  = {results_df.loc[best_model_name, "Test F1"]}')
print(f'   Precision = {results_df.loc[best_model_name, "Test Precision"]}')
print(f'   Recall    = {results_df.loc[best_model_name, "Test Recall"]}')

In [ ]:
# ── 5.3 Confusion Matrix ─────────────────────────────────────────────────────
y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted: No Churn', 'Predicted: Churn'],
    yticklabels=['Actual: No Churn', 'Actual: Churn']
)
plt.title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print('\n📊 Business Interpretation:')
print(f'   True Negatives  (TN) = {tn:,}  → loyal customers correctly identified')
print(f'   False Positives (FP) = {fp:,}  → loyal customers incorrectly targeted for retention (wasted cost)')
print(f'   False Negatives (FN) = {fn:,}  → churners missed (financial risk)')
print(f'   True Positives  (TP) = {tp:,}  → churners correctly detected')

In [ ]:
# ── 5.4 Full Classification Report ───────────────────────────────────────────
print('FULL CLASSIFICATION REPORT')
print('=' * 55)
print(classification_report(y_test, y_pred_best,
                             target_names=['No Churn (0)', 'Churn (1)']))

In [ ]:
# ── 5.5 SHAP Feature Importance ──────────────────────────────────────────────
preprocessor_fitted = best_model.named_steps['preprocessor']
X_test_transformed  = preprocessor_fitted.transform(X_test)

# Get feature names after encoding
num_feature_names = numerical_cols
cat_feature_names = preprocessor_fitted.named_transformers_['cat'].get_feature_names_out(categorical_cols).tolist()
all_feature_names = num_feature_names + cat_feature_names

classifier = best_model.named_steps['classifier']

try:
    explainer = shap.TreeExplainer(classifier)
except:
    explainer = shap.LinearExplainer(classifier, X_test_transformed)

shap_values = explainer.shap_values(X_test_transformed[:200])
if isinstance(shap_values, list):
    shap_values = shap_values[1]

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test_transformed[:200],
    feature_names=all_feature_names,
    max_display=15, show=False
)
plt.title('Feature Importance (SHAP) — Impact on Churn Prediction',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Red = increases churn risk | Blue = decreases churn risk')

In [ ]:
# ── 5.6 Visual model comparison bar chart ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['Test ROC-AUC', 'Test F1']
titles  = ['ROC-AUC Score (test set)', 'F1-Score (test set)']

for ax, metric, title in zip(axes, metrics, titles):
    vals = results_df[metric]
    bars = ax.bar(vals.index, vals.values, color=COLORS, edgecolor='white', linewidth=1.5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontweight='bold')
    best_idx = list(vals.index).index(vals.idxmax())
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

plt.suptitle(f'Model Comparison — Best: {best_model_name} ⭐', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 🔵 PHASE 6 — Deployment

## Saving and Predicting with the Best Model

In [ ]:
# ── 6.1 Save the model ────────────────────────────────────────────────────────
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

model_filename = best_model_name.lower().replace(' ', '_') + '_pipeline.joblib'
save_path = models_dir / model_filename
joblib.dump(best_model, save_path)

print(f'✅ Model saved: {save_path}')
print(f'   File size: {save_path.stat().st_size / 1024:.1f} KB')

In [ ]:
# ── 6.2 Live prediction on a new high-risk customer ───────────────────────────
# Profile: new customer, month-to-month, fiber optic, no security, high charges
new_customer = pd.DataFrame([{
    'SeniorCitizen'  : 0,
    'gender'         : 'Male',
    'Partner'        : 'No',
    'Dependents'     : 'No',
    'tenure'         : 2,
    'PhoneService'   : 'Yes',
    'MultipleLines'  : 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity' : 'No',
    'OnlineBackup'   : 'No',
    'DeviceProtection':'No',
    'TechSupport'    : 'No',
    'StreamingTV'    : 'No',
    'StreamingMovies': 'No',
    'Contract'       : 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod'  : 'Electronic check',
    'MonthlyCharges' : 95.0,
    'TotalCharges'   : 190.0,
}])

# Add engineered features
new_customer['ChargesPerTenureMonth'] = new_customer['MonthlyCharges'] / (new_customer['tenure'] + 1)
new_customer['IsNewCustomer']         = (new_customer['tenure'] <= 6).astype(int)
new_customer['IsLongTerm']            = (new_customer['tenure'] >= 24).astype(int)
new_customer['ServiceBundleScore']    = new_customer[service_cols].apply(lambda x: (x=='Yes').sum(), axis=1)
new_customer['ContractRiskScore']     = new_customer['Contract'].map(contract_risk)
new_customer['TotalChargesGap']       = new_customer['MonthlyCharges'] * new_customer['tenure'] - new_customer['TotalCharges']
new_customer['HighRiskCombo'] = (
    (new_customer['Contract'] == 'Month-to-month') &
    (new_customer['InternetService'] == 'Fiber optic') &
    (new_customer['OnlineSecurity'] == 'No')
).astype(int)

churn_proba = best_model.predict_proba(new_customer)[0, 1]
risk_level  = 'HIGH 🔴' if churn_proba > 0.66 else ('MEDIUM 🟡' if churn_proba > 0.33 else 'LOW 🟢')

print('=' * 55)
print('PREDICTION FOR A NEW CUSTOMER')
print('=' * 55)
print(f'Churn probability : {churn_proba:.1%}')
print(f'Risk level        : {risk_level}')
print()
if churn_proba > 0.33:
    print('💡 CRM Recommendation: Trigger a retention action')
    print('   → Offer a loyalty discount (15% off next 3 months)')
    print('   → Schedule a customer service call within 7 days')
    print('   → Propose migration to a 1-year contract')
else:
    print('✅ Stable customer — no priority action required')

---
# 📋 Project Summary

## Full CRISP-DM Application

| CRISP-DM Phase | Deliverables in this project |
|----------------|------------------------------|
| **1. Business Understanding** | Retention objective defined, success KPIs identified |
| **2. Data Understanding** | 7,043 customers analyzed, distributions visualized, issues identified |
| **3. Data Preparation** | Cleaning, imputation, 7 CRM features engineered, encoding, 70/10/20 split |
| **4. Modeling** | 4 models trained: Logistic Regression, Random Forest, XGBoost, LightGBM |
| **5. Evaluation** | ROC-AUC, F1-score, Precision, Recall, Confusion Matrix, SHAP |
| **6. Deployment** | FastAPI REST endpoint + Streamlit CRM dashboard + saved model |

## Mapping to Course Concepts

| Course concept | Application in this project |
|---------------|-----------------------------|
| **Incomplete data** (Ch.1) | TotalCharges missing → median imputation |
| **Noisy data** (Ch.1) | TotalCharges as string → float coercion |
| **Inconsistent data** (Ch.1) | Churn Yes/No → uniform 0/1 encoding |
| **Supervised learning** (Ch.2) | Labels are known → binary classification |
| **Classification** (Ch.2) | Predicts Churn=0 or Churn=1 per customer |
| **CRISP-DM** (Ch.2) | All 6 phases fully implemented |
| **Data Transformation** (Ch.2) | StandardScaler + OneHotEncoder |
| **Scoring** (Ch.2) | Churn probability score between 0 and 1 per customer |

## 🏆 Results

The best model achieves a **ROC-AUC above 0.80** on the real IBM Telco dataset, which is a strong result for a real-world customer churn problem.

The full system is also deployed as:
- A **REST API** (`http://127.0.0.1:8000/docs`) for integration with CRM systems
- A **Streamlit dashboard** (`http://localhost:8501`) for business users
- An **MLflow tracking** server for experiment management

---
*Project completed as part of the Data Mining course — Dr. Yosra Hajjaji — 2025/2026*